# Fundamentos de Ingeniería de Datos
## BÚSQUEDA: selección del dataset

```
Propósito → criterios → búsqueda en Kaggle → perfil de candidatos → decisión → INGESTA
```

Primero se define **para qué** se usarán los datos y después se buscan los que sirven. Este notebook documenta esa búsqueda. No guarda nada en el proyecto: las descargas de prueba van a una carpeta temporal que se borra al final.

### Propósito

| Elemento | Definición |
|---|---|
| **Pregunta** | ¿Cuánto vale un carro según sus características? |
| **Quién la usa** | Un **agente de ventas** que valúa autos, o una **persona común** que quiere saber cuánto vale su carro o el que piensa comprar |
| **Cómo se responde** | **Modelo:** "¿Cuánto vale mi carro?" → predicción del precio a partir de sus características. **Tablero:** "¿Cuánto vale el carro que quiero comprar?" → gráficas con filtros por marca, año, millaje, etc. |
| **Variable objetivo** | `price` (precio de venta) |

### Criterios que debe cumplir el dataset

| # | Criterio | Por qué |
|---|---|---|
| 1 | Tiene marca, modelo, año, **kilometraje** y características del auto | Son las entradas del modelo y los filtros del tablero |
| 2 | Tiene **precio** | Es la variable que se predice y se analiza |
| 3 | Al menos unos miles de filas | El modelo necesita suficientes ejemplos; cuantas más, mejor |
| 4 | Son **anuncios reales** de venta, no precios de referencia ni subastas | Debe reflejar lo que paga una persona común |
| 5 | Ningún archivo pasa de **100 MB** | Es el límite de GitHub por archivo |

### Configuración del entorno

In [1]:
# ── Importar utils.py ──────────────────────────────────────────────────────

# utils.py contiene las rutas del proyecto y funciones compartidas.
# Vive en Utilidades/utils.py, en la raíz del repositorio.

import sys
import os
sys.path.append(os.path.abspath('../Utilidades'))   # el notebook se ejecuta desde Codigos/

from utils import PROYECTO_BASE, PRIVADO_PATH

print()
print('utils.py importado correctamente')
print(f'Proyecto base: {os.path.basename(PROYECTO_BASE)}/')
print()


utils.py importado correctamente
Proyecto base: ING_BD_ProjectF/



In [2]:
# ── Librerías ──────────────────────────────────────────────────────────────

import glob
import shutil
import zipfile
import tempfile
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

print()
print('Librerías listas')
print()


Librerías listas



### PASO 1 — Configuración de la API de Kaggle

In [3]:
# ── Credenciales de Kaggle: kaggle.json en la carpeta Privado/ ────────────
# 1. Descarga tu token en kaggle.com → Settings → API → Create New Token
# 2. Cópialo a: Privado/kaggle.json  (esa carpeta está en .gitignore, nunca se sube)

KAGGLE_JSON_PATH = f'{PRIVADO_PATH}/kaggle.json'

if os.path.exists(KAGGLE_JSON_PATH):
    os.environ['KAGGLE_CONFIG_DIR'] = PRIVADO_PATH   # carpeta que contiene kaggle.json
    print()
    print('kaggle.json encontrado en Privado/ — usando esa configuración.')
    print()

else:
    # ── Alternativa: ingreso interactivo (solo vive en esta sesión) ──
    from getpass import getpass
    os.environ['KAGGLE_USERNAME'] = input('Usuario de Kaggle: ')
    os.environ['KAGGLE_KEY']      = getpass('API key de Kaggle (no se mostrará en pantalla): ')
    print()
    print('Credenciales cargadas solo en esta sesión (no se guardan en el notebook).')
    print()

import kaggle                                        # se autentica al importarse

print('Conexión con la API de Kaggle lista')
print()


kaggle.json encontrado en Privado/ — usando esa configuración.



Conexión con la API de Kaggle lista



---
## Funciones de búsqueda

Las tres funciones se usan igual para todas las búsquedas. Así cada región se evalúa con los mismos pasos:

| Función | Qué hace |
|---|---|
| `buscar(temas)` | Busca cada tema en Kaggle y regresa una tabla con referencia, título, tamaño, fecha y votos |
| `archivos(refs)` | Lista los archivos de cada dataset candidato con su tamaño, **sin descargarlos** |
| `perfilar(candidatos)` | Descarga cada candidato a una carpeta temporal y mide filas, columnas, marcas, kilometraje y precios distintos |

In [4]:
# ── Funciones de búsqueda ──────────────────────────────────────────────────

def buscar(temas, n=6):
    """Busca cada tema en Kaggle y regresa los primeros n resultados de cada uno."""
    filas = []
    for tema in temas:
        for ds in kaggle.api.dataset_list(search=tema)[:n]:
            filas.append({
                'busqueda'   : tema,
                'ref'        : ds.ref,
                'titulo'     : ds.title,
                'MB'         : round((ds.total_bytes or 0) / 1_048_576, 1),   # tamaño comprimido
                'actualizado': str(ds.last_updated)[:10],
                'votos'      : ds.vote_count,
                'descargas'  : ds.download_count,
            })
    return pd.DataFrame(filas)


def archivos(refs):
    """Lista los archivos de cada dataset (nombre y MB sin comprimir) sin descargarlos."""
    filas = []
    for ref in refs:
        for f in kaggle.api.dataset_list_files(ref).files:
            filas.append({'ref': ref, 'archivo': f.name, 'MB': round(f.total_bytes / 1_048_576, 1)})
    return pd.DataFrame(filas)


def perfilar(candidatos, carpeta):
    """Descarga cada candidato a 'carpeta' (temporal) y mide lo que pide el propósito.

    candidatos: {ref: {'archivo', 'marca', 'precio', 'km', 'lectura'}}
      - archivo: ruta del CSV dentro del dataset
      - marca / precio / km: nombre de esas columnas (km=None si no existe)
      - lectura: argumentos extra para pd.read_csv (separador, encabezado, etc.)
    """
    filas = []
    for ref, c in candidatos.items():
        destino = os.path.join(carpeta, ref.replace('/', '_'))
        os.makedirs(destino, exist_ok=True)

        # Descarga solo el archivo que se va a perfilar (puede llegar como .zip, incluso en subcarpetas)
        kaggle.api.dataset_download_file(ref, c['archivo'], path=destino, quiet=True)
        for z in glob.glob(os.path.join(destino, '**', '*.zip'), recursive=True):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(os.path.dirname(z))
            os.remove(z)

        csv = glob.glob(os.path.join(destino, '**', os.path.basename(c['archivo'])), recursive=True)[0]
        df  = pd.read_csv(csv, low_memory=False, on_bad_lines='skip', **c.get('lectura', {}))

        filas.append({
            'ref'              : ref,
            'filas'            : len(df),
            'columnas'         : df.shape[1],
            'marcas'           : df[c['marca']].nunique(),
            'tiene_km'         : c['km'] is not None,
            'precios_distintos': df[c['precio']].nunique(),
            'MB_csv'           : round(os.path.getsize(csv) / 1_048_576, 1),
        })
        del df
    return pd.DataFrame(filas)


# Carpeta temporal FUERA del proyecto (en la carpeta temporal del sistema): nada de lo que se descargue aquí se sube
TMP = tempfile.mkdtemp(prefix='busqueda_autos_')

print()
print('Funciones listas')
print(f'Descargas de prueba en la carpeta temporal del sistema: {os.path.basename(TMP)}/')
print()


Funciones listas
Descargas de prueba en la carpeta temporal del sistema: busqueda_autos_swge8u4x/



---
## PASO 2 — Búsqueda en México y Latinoamérica

La primera opción fue buscar datos de **México** y después del resto de **Latinoamérica**, para que los precios fueran cercanos al usuario.

In [5]:
# ── Búsqueda: México y Latinoamérica ───────────────────────────────────────

temas_latam = ['cars mexico', 'autos mexico', 'carros usados', 'used cars argentina',
               'mercado libre cars', 'used cars brazil', 'fipe']

resultados_latam = buscar(temas_latam)
resultados_latam

,busqueda,ref,titulo,MB,actualizado,votos,descargas
0,cars mexico,deepcontractor/froza-horizon-5-cars-dataset,FORZA HORIZON 5 CARS DATASET,0.0,2022-01-11,54,1288
1,cars mexico,gabrielvaldesd/2024-mexico-cars,2024-mexico-cars,0.0,2024-08-14,0,39
2,cars mexico,brunopham/formula-e-championship,Formula E Championship Season 1 - Season 10,0.1,2024-02-01,6,187
3,cars mexico,techsalerator/business-funding-data-in-mexico,Business Funding Data in Mexico,0.0,2024-09-14,0,32
4,cars mexico,simongraves/license-plate-dataset,License Plate Dataset OCR 1.9 millions images,3.7,2026-10-09,11,328
5,autos mexico,checoalejandro/autos-consumo-gasolina-mexico,Autos - Consumo Gasolina Mexico,0.3,2018-01-06,24,1952
6,autos mexico,kunwardeepak/youtube-trending-august-2020,(August 2020) Youtube Trending Videos Statistics,1.3,2020-08-22,3,260
7,carros usados,tccleidoprecounico/carros-usados-olx-2021,Carros Usados Olx 2021,2.1,2021-07-19,1,108
8,carros usados,sebastianforero2812/carros,clasificacion,7.3,2022-04-06,2,117
9,used cars argentina,arielras/cars-publications-prices-mercadolibrearg,Cars publications prices - MercadoLibreARG,1.0,2024-02-06,0,139


In [6]:
# ── Archivos de los candidatos de Latinoamérica ────────────────────────────

candidatos_latam = ['gabrielvaldesd/2024-mexico-cars',
                    'arielras/cars-publications-prices-mercadolibrearg',
                    'vagnerbessa/average-car-prices-bazil']

archivos(candidatos_latam)

,ref,archivo,MB
0,gabrielvaldesd/2024-mexico-cars,car_m_v_dataset.csv,0.1
1,arielras/cars-publications-prices-mercadolibrearg,MeliCarPublicactionsCSV/MeliCarPublicactions.csv,2.3
2,arielras/cars-publications-prices-mercadolibrearg,MeliCarPublicactionsJSON/MeliCarPublicactions.json,8.4
3,vagnerbessa/average-car-prices-bazil,fipe_2022.csv,30.9
4,vagnerbessa/average-car-prices-bazil,fipe_cars.csv,62.1


In [7]:
# ── Perfil de los candidatos de Latinoamérica ──────────────────────────────

perfil_latam = perfilar({
    'gabrielvaldesd/2024-mexico-cars': {
        'archivo': 'car_m_v_dataset.csv', 'marca': 'brand', 'precio': 'max', 'km': None,
        'lectura': {'header': 1}},                               # la fila 0 agrupa columnas
    'arielras/cars-publications-prices-mercadolibrearg': {
        'archivo': 'MeliCarPublicactionsCSV/MeliCarPublicactions.csv', 'marca': 'brand',
        'precio': 'price', 'km': 'km', 'lectura': {'sep': ';', 'encoding': 'utf-8-sig'}},
    'vagnerbessa/average-car-prices-bazil': {
        'archivo': 'fipe_2022.csv', 'marca': 'brand', 'precio': 'avg_price_brl', 'km': None},
}, TMP)

perfil_latam

Dataset URL: https://www.kaggle.com/datasets/gabrielvaldesd/2024-mexico-cars


Dataset URL: https://www.kaggle.com/datasets/arielras/cars-publications-prices-mercadolibrearg


Dataset URL: https://www.kaggle.com/datasets/vagnerbessa/average-car-prices-bazil


,ref,filas,columnas,marcas,tiene_km,precios_distintos,MB_csv
0,gabrielvaldesd/2024-mexico-cars,241,41,9,False,219,0.1
1,arielras/cars-publications-prices-mercadolibrearg,10191,25,8,True,2568,2.3
2,vagnerbessa/average-car-prices-bazil,290275,12,86,False,143674,30.9


**Resultado en Latinoamérica:** ningún candidato cumple todos los criterios.

| Dataset | Qué es | Por qué se descarta |
|---|---|---|
| `gabrielvaldesd/2024-mexico-cars` | Precios de autos **nuevos** en México (MXN) | Solo unos cientos de filas (criterio 3) y sin kilometraje (criterio 1) |
| `arielras/cars-publications-prices-mercadolibrearg` | Anuncios de MercadoLibre Argentina | Unas 10 mil filas, pero solo 8 marcas y precios mezclados en ARS y USD, con mucha inflación |
| `vagnerbessa/average-car-prices-bazil` | Tabla FIPE de Brasil | Son **precios promedio de referencia**, no anuncios (criterio 4), y no tiene kilometraje (criterio 1) |

Fuera de Kaggle solo se encontraron servicios de pago (extractores de Kavak, Seminuevos y Autocosmos). Por eso la búsqueda se amplió a **Estados Unidos**, donde los datasets de anuncios son grandes y tienen kilometraje.

---
## PASO 3 — Búsqueda en Estados Unidos

In [8]:
# ── Búsqueda: Estados Unidos ───────────────────────────────────────────────

temas_us = ['us used cars', 'used car prices', 'craigslist cars', 'car auction prices',
            'cars.com', 'cargurus']

resultados_us = buscar(temas_us)
resultados_us

,busqueda,ref,titulo,MB,actualizado,votos,descargas
0,us used cars,austinreese/craigslist-carstrucks-data,Used Cars Dataset,248.4,2026-11-05,1702,124764
1,us used cars,ananaymital/us-used-cars-dataset,US Used cars dataset,2181.2,2020-09-21,98,10278
2,us used cars,lepchenkov/usedcarscatalog,Used-cars-catalog,0.8,2019-12-02,169,35994
3,us used cars,thedevastator/uncovering-factors-that-affect-used-car-pr...,Used Cars,18.2,2022-12-06,100,10709
4,us used cars,avikasliwal/used-cars-price-prediction,Used Cars Price Prediction,0.2,2019-06-25,267,34938
5,us used cars,mrmars1010/cars-india-pre-owned,Used Cars Dataset,0.1,2024-09-29,60,3577
6,used car prices,tunguz/used-car-auction-prices,Used Car Auction Prices,19.0,2021-05-18,126,9575
7,used car prices,sujay1844/used-car-prices,Used Car Prices,0.1,2023-10-24,51,4851
8,used car prices,talhabarkaatahmad/pakistan-used-car-prices-2023,Pakistan Used Car Prices 2023,2.7,2023-09-29,38,1878
9,used car prices,anthonytherrien/extended-dataset-for-used-car-prices-reg...,Extended Dataset for Used Car Prices Regression,0.3,2024-09-08,44,1805


In [9]:
# ── Archivos de los candidatos de Estados Unidos ───────────────────────────

candidatos_us = ['andreinovikov/used-cars-dataset',
                 'tunguz/used-car-auction-prices',
                 'austinreese/craigslist-carstrucks-data',
                 'rebrowser/carguruscom-dataset']

archivos_us = archivos(candidatos_us)
archivos_us.groupby('ref').head(3)          # CarGurus tiene un archivo por día: se muestran 3

,ref,archivo,MB
0,andreinovikov/used-cars-dataset,cars.csv,138.5
1,tunguz/used-car-auction-prices,car_prices.csv,85.0
2,austinreese/craigslist-carstrucks-data,vehicles.csv,1109.5
3,rebrowser/carguruscom-dataset,README.md,0.0
4,rebrowser/carguruscom-dataset,car-listings/data/2026-10-11.csv,3.6
5,rebrowser/carguruscom-dataset,car-listings/data/2026-10-11.parquet,0.7


`austinreese/craigslist-carstrucks-data` se descarta sin descargarlo: su CSV pesa más de **1 GB**, más de 10 veces el límite de GitHub (criterio 5). Los otros tres se perfilan:

In [10]:
# ── Perfil de los candidatos de Estados Unidos ─────────────────────────────

perfil_us = perfilar({
    'andreinovikov/used-cars-dataset': {
        'archivo': 'cars.csv', 'marca': 'manufacturer', 'precio': 'price', 'km': 'mileage'},
    'tunguz/used-car-auction-prices': {
        'archivo': 'car_prices.csv', 'marca': 'make', 'precio': 'sellingprice', 'km': 'odometer'},
    'rebrowser/carguruscom-dataset': {
        'archivo': 'car-listings/data/2026-10-11.csv', 'marca': 'make', 'precio': 'price', 'km': 'mileage'},
}, TMP)

perfil_us

Dataset URL: https://www.kaggle.com/datasets/andreinovikov/used-cars-dataset


Dataset URL: https://www.kaggle.com/datasets/tunguz/used-car-auction-prices


Dataset URL: https://www.kaggle.com/datasets/rebrowser/carguruscom-dataset


,ref,filas,columnas,marcas,tiene_km,precios_distintos,MB_csv
0,andreinovikov/used-cars-dataset,762091,20,30,True,63464,138.5
1,tunguz/used-car-auction-prices,558811,16,96,True,1887,85.0
2,rebrowser/carguruscom-dataset,1000,64,35,True,1,3.6


**Lectura del perfil:**

| Dataset | Qué es | Decisión |
|---|---|---|
| `andreinovikov/used-cars-dataset` (Cars.com) | **Anuncios de venta al público** (2023), autos de 1915 a 2024 | ✅ **Elegido** |
| `tunguz/used-car-auction-prices` | Subastas **mayoristas** entre concesionarios (2014-2015) | ❌ No es lo que paga una persona común (criterio 4) y los datos son viejos |
| `rebrowser/carguruscom-dataset` | Muestra gratuita de CarGurus (1 000 filas por día) | ❌ El precio está **oculto**: tiene un solo valor distinto en todas las filas (criterio 2) |

---
## PASO 4 — Decisión

**Dataset elegido:** [`andreinovikov/used-cars-dataset`](https://www.kaggle.com/datasets/andreinovikov/used-cars-dataset), etiqueta **`autos`**.

| Criterio | ¿Cumple? | Evidencia |
|---|---|---|
| 1. Columnas de la pregunta | ✅ | Marca, modelo, año, millaje, motor, transmisión, tracción, combustible, rendimiento, accidentes, un solo dueño, uso personal |
| 2. Precio | ✅ | `price` en USD |
| 3. Miles de filas | ✅ | Más de 760 mil anuncios de 30 marcas |
| 4. Anuncios reales | ✅ | Anuncios de Cars.com |
| 5. Menos de 100 MB por archivo | ⚠️ | El CSV completo pesa unos 140 MB → **se toma una muestra** |

### Cómo se cumple el criterio 5: muestra estratificada

En **INGESTA** se descarga el CSV completo y se toma una **muestra de 200 000 filas estratificada por marca**, con semilla fija (`random_state=42`):

- **Estratificada:** cada marca conserva la misma proporción que en el dataset completo. Así la muestra representa bien al mercado y el modelo ve todas las marcas.
- **Semilla fija:** cualquiera que ejecute el notebook obtiene exactamente las mismas filas.
- **200 000 filas:** son suficientes para el modelo y el tablero. Con ellas, el CSV de la muestra y `base_datos.db` quedan por debajo de 100 MB.
- **La muestra no limpia nada:** las filas se copian tal como vienen. RAW sigue siendo crudo, y toda la limpieza se hace en TRANSFORMACION.
- El CSV completo se queda solo en la computadora local, porque está en `.gitignore`.

In [11]:
# ── Borrar las descargas de prueba ─────────────────────────────────────────

shutil.rmtree(TMP, ignore_errors=True)

print()
print(f'Carpeta temporal borrada: {os.path.basename(TMP)}/')
print('Siguiente paso: INGESTA.ipynb')
print()


Carpeta temporal borrada: busqueda_autos_swge8u4x/
Siguiente paso: INGESTA.ipynb

